In [1]:
import requests

api_url = "https://115.159.159.82/api/devices?"

r = requests.get(
    api_url,
    params={
        "page":1,"page_size":50
    },
    timeout=10
)

print("status:", r.status_code)
print("url:", r.url)
print("content-type:", r.headers.get("Content-Type"))

status: 200
url: https://115.159.159.82/api/devices?page=1&page_size=50
content-type: application/json


In [2]:
data = r.json()

print(type(data))
print(data.keys())

<class 'dict'>
dict_keys(['page', 'page_size', 'total', 'items'])


In [3]:
print(data)

{'page': 1, 'page_size': 50, 'total': 5000, 'items': [{'device_id': 'D00000001', 'company_id': 'C000001', 'device_name': '循环泵-000001-001', 'device_type': 'pump', 'model': 'SP-PUM-800', 'zone': '智造一区', 'install_date': '2021-09-19', 'rated_power_kw': 71.2, 'sampling_interval_s': 30, 'protocol': 'mqtt', 'status': 'running', 'updated_at': '2026-09-01T03:58:00'}, {'device_id': 'D00000002', 'company_id': 'C000001', 'device_name': '循环泵-000001-002', 'device_type': 'pump', 'model': 'SP-PUM-500', 'zone': '智造一区', 'install_date': '2016-03-06', 'rated_power_kw': 73.8, 'sampling_interval_s': 60, 'protocol': 'mqtt', 'status': 'running', 'updated_at': '2026-09-08T05:44:00'}, {'device_id': 'D00000003', 'company_id': 'C000001', 'device_name': '空压机-000001-003', 'device_type': 'compressor', 'model': 'SP-COM-200', 'zone': '智造一区', 'install_date': '2016-12-18', 'rated_power_kw': 248.6, 'sampling_interval_s': 30, 'protocol': 'modbus', 'status': 'running', 'updated_at': '2026-08-29T04:01:00'}, {'device_id': 'D

In [4]:
print(data["items"][0])

{'device_id': 'D00000001', 'company_id': 'C000001', 'device_name': '循环泵-000001-001', 'device_type': 'pump', 'model': 'SP-PUM-800', 'zone': '智造一区', 'install_date': '2021-09-19', 'rated_power_kw': 71.2, 'sampling_interval_s': 30, 'protocol': 'mqtt', 'status': 'running', 'updated_at': '2026-09-01T03:58:00'}


In [5]:
total = data["total"]
page_size = data["page_size"]

total_pages = (total + page_size - 1) // page_size

print("total:", total)
print("page_size:", page_size)
print("total_pages:", total_pages)

total: 5000
page_size: 50
total_pages: 100


In [6]:
devices = []

for page in range(1, total_pages + 1):
    r = requests.get(
        api_url,
        params={
            # 填写 page 和 page_size
            "page": page,
            "page_size":page_size
        },
        timeout=10
    )

    page_data = r.json()
    devices.extend(page_data["items"])
    # 将本页设备加入 devices
    # TODO

print("device count:", len(devices))

device count: 5000


In [7]:
from bs4 import BeautifulSoup

base_url = "https://115.159.159.82"
current_url = base_url + "/companies"

r = requests.get(current_url, timeout=10)
soup = BeautifulSoup(r.text, "html.parser")

In [8]:
cards = soup.select(".company-card")
print("本页企业数：", len(cards))

本页企业数： 20


In [9]:
cards = soup.select(".company-card")

card = cards[0]

In [10]:
details = {}

for item in card.select("dl > div"):
    key = item.select_one("dt").get_text(strip=True)
    value = item.select_one("dd").get_text(strip=True)
    details[key] = value

print(details)

{'所属行业': '汽车零部件', '所在区域': '智造一区', '员工规模': '88 人', '设备数量': '6 台'}


In [11]:
cards = soup.select(".company-card")
print("本页企业数：", len(cards))

本页企业数： 20


In [12]:
cards = soup.select(".company-card")
card = cards[0]
company_id = card.select_one(".company-card-top .mono").get_text(strip=True)
name = card.select_one("h2 a").get_text(strip=True)
status = card.select_one(".company-card-top .badge").get_text(strip=True)


details = {}
for item in card.select("dl > div"):
    key = item.select_one("dt").get_text(strip=True)
    value = item.select_one("dd").get_text(strip=True)
    details[key] = value

industry = details["所属行业"]
zone = details["所在区域"]
device_count = int(details["设备数量"].replace("台","").strip())

In [13]:
from urllib.parse import urljoin

In [14]:
next_link = soup.select_one("a:contains('下一页')")
next_href = next_link.get("href")
next_url = urljoin(
    current_url,
    next_href
)


D:\ANACONDA\Lib\site-packages\soupsieve\css_parser.py:856: FutureWarning: The pseudo class ':contains' is deprecated, ':-soup-contains' should be used moving forward.
  warnings.warn(  # noqa: B028


In [15]:
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin


In [16]:
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin

base_url = "https://115.159.159.82"
current_url = base_url + "/companies"

# ✅ 改成 companies，不再用 all_companies
companies = []
max_collect_page = 3

for _ in range(max_collect_page):
    print(f"正在请求：{current_url}")
    resp = requests.get(current_url, timeout=10)
    soup = BeautifulSoup(resp.text, "html.parser")

    cards = soup.select(".company-card")
    for card in cards:
        company_id = card.select_one(".company-card-top .mono").get_text(strip=True)
        name = card.select_one("h2 a").get_text(strip=True)
        status = card.select_one(".company-card-top .badge").get_text(strip=True)

        details = {}
        for item in card.select("dl > div"):
            key = item.select_one("dt").get_text(strip=True)
            value = item.select_one("dd").get_text(strip=True)
            details[key] = value

        industry = details["所属行业"]
        zone = details["所在区域"]
        device_count = int(details["设备数量"].replace("台","").strip())

        companies.append({
            "company_id": company_id,
            "name": name,
            "status": status,
            "industry": industry,
            "zone": zone,
            "device_count": device_count
        })
    print(f"本页采集数量：{len(cards)}")

    next_link = soup.find("a", string="下一页")
    if next_link is not None:
        next_href = next_link.get("href")
        current_url = urljoin(current_url, next_href)
    else:
        break

print(f"\n全部采集完成，Company总数：{len(companies)}")


正在请求：https://115.159.159.82/companies


本页采集数量：20
正在请求：https://115.159.159.82/companies?page=2


本页采集数量：20
正在请求：https://115.159.159.82/companies?page=3


本页采集数量：20

全部采集完成，Company总数：60


In [17]:
print("companies:", len(companies))
print(
    "unique company_id:",
    len(set(x["company_id"] for x in companies))
)

print("devices:", len(devices))
print(
    "unique device_id:",
    len(set(x["device_id"] for x in devices))
)

companies: 60
unique company_id: 60
devices: 5000
unique device_id: 5000


---

## 任务四：Company–Device 一致性检查


In [18]:
from collections import Counter

device_counts = Counter(
    x["company_id"] for x in devices
)

mismatch_count = 0
for company in companies:
    cid = company["company_id"]
    expected = company["device_count"]
    actual = device_counts.get(cid, 0)
    if expected != actual:
        mismatch_count += 1

print("Company-Device 不一致企业数:", mismatch_count)


Company-Device 不一致企业数: 0


---

## 任务五：保存采集结果

验证通过后，把结果保存到 Notebook 同级的  目录。


In [19]:
import json
from pathlib import Path

data_dir = Path("data")
data_dir.mkdir(exist_ok=True)

with open(data_dir / "devices.json", "w", encoding="utf-8") as f:
    json.dump(devices, f, ensure_ascii=False, indent=2)

with open(data_dir / "companies.json", "w", encoding="utf-8") as f:
    json.dump(companies, f, ensure_ascii=False, indent=2)

print("数据已保存到 data/ 目录")


数据已保存到 data/ 目录
